# EdgeMoE on Kaggle — end-to-end

This notebook runs a real HuggingFace MoE model on a Kaggle T4 (16 GB VRAM) by streaming expert weights from disk.

We use **OLMoE-1B-7B** (~7B params, 64 experts × 16 layers) as the demo model — it downloads in ~3 minutes on Kaggle and fits comfortably in VRAM after the split.

**Settings to enable before running:**
- *Settings → Accelerator → GPU T4 x2* (one is enough; two just gives headroom)
- *Settings → Internet → On*

**How to install the code** (pick one):
- **A.** Upload the `edgemoe/` folder as a Kaggle Dataset named `edgemoe-code`, then run cell 1.
- **B.** Clone from GitHub (once you've pushed the repo) — replace cell 1 accordingly.

## 1. Install EdgeMoE

In [ ]:
# Option A — installed from a Kaggle Dataset
import subprocess, sys, os
DATASET = '/kaggle/input/edgemoe-code'
if os.path.isdir(DATASET):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', DATASET])
else:
    # Option B — fall back to cloning from your GitHub
    # !git clone https://github.com/<you>/edgemoe.git && pip install -q -e edgemoe
    raise SystemExit('Upload the repo as a Kaggle Dataset named edgemoe-code, or edit cell to clone from GitHub.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers>=4.43', 'safetensors', 'huggingface_hub', 'hf_xet', 'accelerate'])

## 2. Split the HF model into per-expert files

This runs `tools/split_experts.py` against OLMoE-1B-7B. The splitter streams tensors one-by-one from safetensors shards, so RAM stays under ~1 GB even for big MoE models.

Expected: ~5 minutes (download) + ~3 minutes (quantise) on Kaggle.

In [ ]:
from tools.split_experts import split_model

SPLIT_DIR = '/kaggle/working/olmoe-split'
split_model(
    hf_model='allenai/OLMoE-1B-7B-0924-Instruct',
    output_dir=SPLIT_DIR,
    quant_mode='q4',            # 'bitnet' for 1.58-bit ternary experts
)

## 3. Load EdgeMoE

In [ ]:
from edgemoe import EdgeMoE

engine = EdgeMoE(
    model=SPLIT_DIR,
    backend='local',
    vram_budget_gb=10.0,        # T4 has 16 GB, backbone + buffers take ~4
    ram_buffer_gb=10.0,
)

## 4. Generate

In [ ]:
print(engine.generate(
    'Explain why the sky is blue in two sentences.',
    max_tokens=80,
    temperature=0.7,
))

## 5. Benchmark + inspect

In [ ]:
import json
stats = engine.benchmark(prompt='The quick brown fox', num_tokens=64)
print(json.dumps(stats, indent=2))

In [ ]:
print('cache :', engine.cache.stats())
print('pref  :  in_flight=', len(engine.prefetcher.in_flight), 'buffer=', len(engine.prefetcher.buffer))
engine.close()

## 6. Next: Qwen3-30B-A3B

Swap the model in cell 2:
```python
split_model('Qwen/Qwen3-30B-A3B-Instruct-2507', SPLIT_DIR, quant_mode='bitnet')
```
At `bitnet` (1.58-bit ternary) the 30B model is ~6 GB of expert files + ~1 GB backbone. Fits in Kaggle's 20 GB of `/kaggle/working`.

Pass `backend='gdrive'` once you've uploaded the split directory to Drive — experts then stream from the user's 2 TB Drive instead of local disk.